# Session 6: API'er og HTTP-klienter — Byg en klient til din API

## Dagens undervisning

### HTTP-grundbegreber
- Request: metode (GET/POST), URL, headers, body
- Response: statuskode, headers, body (JSON)
- REST API'er: ressourcer, endpoints, autentificering

### httpx — den moderne HTTP-klient

`httpbin.org/get` sender din request tilbage som JSON — så kan du se præcis, hvad du sendte.

In [ ]:
import httpx

# Synkront
response = httpx.get("https://httpbin.org/get", headers={"Authorization": "Bearer TOKEN"})
data = response.json()
print(response.status_code)
data

### Hold hemmeligheder ude af koden

I den her mappe ligger tre filer:

- `.env` — her skriver du din nøgle fra Mistral: `MISTRAL_API_KEY=sk-...`
- `.env.example` — viser, hvilke variabler `.env` skal indeholde (uden rigtige nøgler). Den bliver committet.
- `.gitignore` — sørger for, at `.env` aldrig bliver committet

Indsæt din nøgle i `.env`, og kør så cellen nedenfor.

In [ ]:
# .env-fil (commit den aldrig!)
# MISTRAL_API_KEY=sk-...

from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.environ["MISTRAL_API_KEY"]

- `.env` skal i `.gitignore`
- Brug `.env.example` til at dokumentere, hvilke variabler der skal bruges

### POST med JSON-body — kald Mistral

In [ ]:
# POST med JSON-body
response = httpx.post(
    "https://api.mistral.ai/v1/chat/completions",
    headers={"Authorization": f"Bearer {api_key}"},
    json={"model": "mistral-small-latest", "messages": [{"role": "user", "content": "Hello"}]},
)
print(response.status_code)
response.json()["choices"][0]["message"]["content"]

### Streaming-svar

In [ ]:
url = "https://api.mistral.ai/v1/chat/completions"
headers = {"Authorization": f"Bearer {api_key}"}
payload = {
    "model": "mistral-small-latest",
    "messages": [{"role": "user", "content": "Skriv et kort digt om Python"}],
    "stream": True,
}

with httpx.stream("POST", url, json=payload, headers=headers) as response:
    for chunk in response.iter_text():
        print(chunk, end="", flush=True)

- Hvorfor streaming? Oplevet hastighed — de første tokens dukker op med det samme
- Server-Sent Events (SSE)-formatet: `data: {...}\n\n`